In [ ]:
#Import Library
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from google.colab import files

# Load CSV file
df = pd.read_csv('clean_dataset.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102895 entries, 0 to 102894
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   organization_id         102895 non-null  object 
 1   activity_name           102895 non-null  object 
 2   timestamp               102895 non-null  object 
 3   converted               102895 non-null  bool   
 4   converted_at            22223 non-null   object 
 5   trial_start             102895 non-null  object 
 6   trial_end               102895 non-null  object 
 7   days_since_trial_start  102895 non-null  int64  
 8   trial_duration          102895 non-null  int64  
 9   time_to_convert         22223 non-null   float64
 10  is_first_week           102895 non-null  bool   
 11  week_of_trial           102895 non-null  int64  
dtypes: bool(2), float64(1), int64(3), object(6)
memory usage: 8.0+ MB


## Organization-Level Feature Engineering
To analyze conversion behavior, event-level data was aggregated into organization-level metrics. This allows us to compare engagement patterns between converting and non-converting trialists.

The following features were created:

Total events: Overall product usage volume Active days: Frequency of engagement Max active day: Duration of engagement within trial period Conversion status: Target variable for analysis

In [ ]:
org_df = df.groupby('organization_id').agg({
    'activity_name': 'count',
    'timestamp': 'nunique',
    'converted': 'max'
}).reset_index()

org_df.rename(columns={
    'activity_name': 'total_events',
    'timestamp': 'active_days'
}, inplace=True)

In [ ]:
# Conversion rate
conversion_rate = org_df['converted'].mean()
print("Conversion Rate:", conversion_rate)

Conversion Rate: 0.21325051759834368


Overall conversion rate across trial organizations is 21%, providing a baseline for evaluating product performance.

Measurable Goals


In [ ]:
df_week1 = df[df['week_of_trial'] == 1]
goal_df = df_week1.groupby('organization_id').agg({
    'activity_name': lambda x: list(x)
}).reset_index()

In [ ]:
#Create a goal flag
def define_goals(actions):
    return pd.Series({
        'goal_schedule_created': 'Scheduling.Shift.Created' in actions,
        'goal_schedule_viewed': 'Mobile.Schedule.Loaded' in actions,
        'goal_team_engaged': 'Communication.Message.Created' in actions,
        'goal_execution': 'PunchClock.PunchedIn' in actions,
        'goal_admin': any(a in actions for a in [
            'Timesheets.BulkApprove.Confirmed',
            'Scheduling.Shift.Approved'
        ])
    })

goal_flags = goal_df['activity_name'].apply(define_goals)
goal_df = pd.concat([goal_df['organization_id'], goal_flags], axis=1)

In [ ]:
#merge with conversion
goal_df = org_df[['organization_id', 'converted']].merge(
    goal_df,
    on='organization_id',
    how='left'
).fillna(False)

/tmp/ipykernel_1302/2983438388.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False)


In [ ]:
# Define goal columns
goal_columns = [
    'goal_schedule_created',
    'goal_schedule_viewed',
    'goal_team_engaged',
    'goal_execution',
    'goal_admin'
]

# Create goals_completed
goal_df['goals_completed'] = goal_df[goal_columns].sum(axis=1)

# Run Your Analysis
pd.crosstab(goal_df['goals_completed'], goal_df['converted'], normalize='index')

converted,False,True
goals_completed,,
0,0.800000,0.200000
1,0.788168,0.211832
2,0.792453,0.207547
3,0.745283,0.254717
4,0.769231,0.230769
5,0.888889,0.111111


Analysis of individual trial goals shows only marginal differences in conversion rates, indicating that no single action is sufficient to drive conversion. While actions such as shift creation and schedule viewing show slight positive associations, their individual impact is limited.

This suggests that conversion is better explained by a combination of behaviors rather than isolated actions. Organizations that engage across multiple core features are more likely to convert, reinforcing the importance of breadth and consistency of usage.

In [ ]:
df_week1 = df[df['week_of_trial'] == 1].copy()

engagement = df_week1.groupby('organization_id').agg({
    'activity_name': 'count',
    'days_since_trial_start': 'nunique'
}).rename(columns={
    'activity_name': 'total_events_w1',
    'days_since_trial_start': 'active_days_w1'
}).reset_index()

In [ ]:
#Merge Into goal_df
goal_df= goal_df.merge(
    engagement,
    on='organization_id',
    how='left'
).fillna(0)

2. Activation Rate

In [ ]:
goal_df['activated'] = (
    (goal_df['goals_completed'] >= 2) &
    (goal_df['total_events_w1'] >= 10) &
    (goal_df['active_days_w1'] >= 3)
)

activation_rate = goal_df['activated'].mean()
print("Activation Rate:", round(activation_rate, 3))

Activation Rate: 0.142


Only 14% of trialists reached activation, indicating that many users do not experience core product value during the trial.

In [ ]:
pd.crosstab(goal_df['activated'], goal_df['converted'], normalize='index')

converted,False,True
activated,,
False,0.790109,0.209891
True,0.766423,0.233577


Activated users show a slightly higher conversion rate compared to non-activated users, increasing from ~21% to ~23%. While this suggests some positive relationship, the difference is modest, indicating that the current activation definition captures only part of the factors influencing conversion.

In [ ]:
#Total Events
goal_df.groupby('converted')['total_events_w1'].mean()

,total_events_w1
converted,
False,33.475000
True,37.737864


Converters demonstrate higher engagement in the first week, averaging approximately 38 events compared to 33 for non-converters. Organizations that convert tend to engage more actively during the first week of the trial, performing a higher number of product interactions compared to non-converters.

In [ ]:
#Active Days
goal_df.groupby('converted')['active_days_w1'].mean()

,active_days_w1
converted,
False,1.650000
True,1.684466


The number of active days in the first week is nearly identical for converters and non-converters, suggesting that consistency of usage alone does not influence conversion outcomes.

In [ ]:
pd.crosstab(goal_df['goals_completed'], goal_df['converted'], normalize='index')

converted,False,True
goals_completed,,
0,0.800000,0.200000
1,0.788168,0.211832
2,0.792453,0.207547
3,0.745283,0.254717
4,0.769231,0.230769
5,0.888889,0.111111


Conversion rates show a modest increase for organizations completing around three core product actions, reaching approximately 25%. However, this relationship is not consistent across all levels, suggesting that completing additional actions beyond a certain point does not necessarily lead to higher conversion.